In [2]:
import pennylane as qml
from pennylane import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Define the device
n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)

# Define quantum circuit (ansatz)
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Define weight shapes
weight_shapes = {"weights": (3, n_qubits, 3)}

# Wrap as a QNode
qnode = qml.QNode(quantum_circuit, dev, interface="torch", diff_method="backprop")

# Define a PyTorch-compatible QNN layer
class QNNLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

    def forward(self, x):
        return self.qlayer(x)

# Full Hybrid Model (1 classical layer + 1 QNN layer + classifier)
class HybridQNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)     # Classical preprocessing
        self.qnn = QNNLayer()
        self.fc2 = nn.Linear(2, 1)     # Final classifier

    def forward(self, x):
        x = self.fc1(x)
        x = self.qnn(x)
        x = self.fc2(x)
        return torch.sigmoid(x)

# Create toy dataset: 2D binary classification
def generate_data(n_samples=100):
    X = np.random.randn(n_samples, 2)
    y = (X[:, 0] * X[:, 1] > 0).astype(float)  # Label: 1 if x and y have same sign
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)

X, y = generate_data(200)

# Train/test split
X_train, X_test = X[:150], X[150:]
y_train, y_test = y[:150], y[150:]

# Instantiate model
model = HybridQNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
epochs = 30
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()
    
    acc = ((output > 0.5) == y_train).float().mean()
    print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Accuracy={acc.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    test_output = model(X_test)
    test_acc = ((test_output > 0.5) == y_test).float().mean()
    print(f"\nTest Accuracy: {test_acc.item():.4f}")


Epoch 1: Loss=0.7117, Accuracy=0.3933
Epoch 2: Loss=0.7082, Accuracy=0.4000
Epoch 3: Loss=0.7050, Accuracy=0.3933
Epoch 4: Loss=0.7020, Accuracy=0.3933
Epoch 5: Loss=0.6991, Accuracy=0.3933
Epoch 6: Loss=0.6963, Accuracy=0.4067
Epoch 7: Loss=0.6936, Accuracy=0.3933
Epoch 8: Loss=0.6909, Accuracy=0.4000
Epoch 9: Loss=0.6883, Accuracy=0.4133
Epoch 10: Loss=0.6856, Accuracy=0.4400
Epoch 11: Loss=0.6830, Accuracy=0.4867
Epoch 12: Loss=0.6804, Accuracy=0.4800
Epoch 13: Loss=0.6777, Accuracy=0.4933
Epoch 14: Loss=0.6750, Accuracy=0.5467
Epoch 15: Loss=0.6724, Accuracy=0.5867
Epoch 16: Loss=0.6697, Accuracy=0.6133
Epoch 17: Loss=0.6670, Accuracy=0.5933
Epoch 18: Loss=0.6644, Accuracy=0.6067
Epoch 19: Loss=0.6617, Accuracy=0.6267
Epoch 20: Loss=0.6591, Accuracy=0.6267
Epoch 21: Loss=0.6564, Accuracy=0.6400
Epoch 22: Loss=0.6537, Accuracy=0.6400
Epoch 23: Loss=0.6509, Accuracy=0.6667
Epoch 24: Loss=0.6480, Accuracy=0.6467
Epoch 25: Loss=0.6450, Accuracy=0.6467
Epoch 26: Loss=0.6419, Accuracy=0.